GridWorld Environment   

In [8]:
!pip install copier


Defaulting to user installation because normal site-packages is not writeable
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
Using cached jinja2-3.1.6-py3-none-any.whl (134 kB)

   ---------------------------------------- 0/7 [funcy]
   ----- ---------------------------------- 1/7 [plumbum]
   ----- ---------------------------------- 1/7 [plumbum]
   ----- ---------------------------------- 1/7 [plumbum]
   ----- ---------------------------------- 1/7 [plumbum]
   ----- ---------------------------------- 1/7 [plumbum]
   ----- ---------------------------------- 1/7 [plumbum]
  Attempting uninstall: jinja2
   ----- ---------------------------------- 1/7 [plumbum]
    Found existing installation: Jinja2 3.1.4
   ----- ---------------------------------- 1/7 [plumbum]
    Uninstalling Jinja2-3.1.4:
   ----- ---------------------------------- 1/7 [plumbum]
      Successfully uninstalled Jinja2-3.1.4
   ----- ---------------------------------- 1/7 [plumbum]
   ----------- ---

In [9]:
# !pip install gymnasium numpy

import numpy as np
import gymnasium as gym
from typing import Optional

print("gymnasium version:", gym.__version__)
print("numpy version:", np.__version__)

gymnasium version: 1.3.0
numpy version: 2.1.2


In [11]:
class GridWorldEnv(gym.Env):

    def __init__(self, size: int = 5):
        # The size of the square grid (5x5 by default)
        self.size = size

        # Initialize positions - will be set randomly in reset()
        # Using -1,-1 as "uninitialized" state
        self._agent_location = np.array([-1, -1], dtype=np.int32)
        self._target_location = np.array([-1, -1], dtype=np.int32)

        # Define what the agent can observe
        # Dict space gives us structured, human-readable observations
        self.observation_space = gym.spaces.Dict(
            {
                "agent": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),   # [x, y] coordinates
                "target": gym.spaces.Box(0, size - 1, shape=(2,), dtype=int),  # [x, y] coordinates
            }
        )

        # Define what actions are available (4 directions)
        self.action_space = gym.spaces.Discrete(4)

        # Map action numbers to actual movements on the grid
        # This makes the code more readable than using raw numbers
        self._action_to_direction = {
            0: np.array([0, 1]),   # Move right (column + 1)
            1: np.array([-1, 0]),  # Move up (row - 1)
            2: np.array([0, -1]),  # Move left (column - 1)
            3: np.array([1, 0]),   # Move down (row + 1)
        }

    def _get_obs(self):
        """Convert internal state to observation format.

        Returns:
            dict: Observation with agent and target positions
        """
        return {"agent": self._agent_location, "target": self._target_location}    

    def _get_info(self):
        """Compute auxiliary information for debugging.

        Returns:
            dict: Info with distance between agent and target
        """
        return {
            "distance": np.linalg.norm(
                self._agent_location - self._target_location, ord=1
            )
        }

    
    def reset(self, seed: Optional[int] = None, options: Optional[dict] = None):
        """Start a new episode.

        Args:
            seed: Random seed for reproducible episodes
            options: Additional configuration (unused in this example)

        Returns:
            tuple: (observation, info) for the initial state
        """
        # IMPORTANT: Must call this first to seed the random number generator
        super().reset(seed=seed)

        # Randomly place the agent anywhere on the grid
        self._agent_location = self.np_random.integers(0, self.size, size=2, dtype=int)

        # Randomly place target, ensuring it's different from agent position
        self._target_location = self._agent_location
        while np.array_equal(self._target_location, self._agent_location):
            self._target_location = self.np_random.integers(
                0, self.size, size=2, dtype=int
            )

        observation = self._get_obs()
        info = self._get_info()

        return observation, info


    def step(self, action):
        """Execute one timestep within the environment.

        Args:
            action: The action to take (0-3 for directions)

        Returns:
            tuple: (observation, reward, terminated, truncated, info)
        """
        # Map the discrete action (0-3) to a movement direction
        direction = self._action_to_direction[action]

        # Update agent position, ensuring it stays within grid bounds
        # np.clip prevents the agent from walking off the edge
        self._agent_location = np.clip(
            self._agent_location + direction, 0, self.size - 1
        )

        # Check if agent reached the target
        terminated = np.array_equal(self._agent_location, self._target_location)

        # We don't use truncation in this simple environment
        # (could add a step limit here if desired)
        truncated = False

        # Simple reward structure: +1 for reaching target, 0 otherwise
        # Alternative: could give small negative rewards for each step to encourage efficiency
        reward = 1 if terminated else 0

        observation = self._get_obs()
        info = self._get_info()

        return observation, reward, terminated, truncated, info

In [13]:
env = GridWorldEnv(size=5)

obs, info = env.reset(seed=42)
print("Initial observation:", obs)
print("Initial info:", info)

obs, reward, terminated, truncated, info = env.step(0)  # move right
print("\nAfter one step (action=0, 'right'):")
print("obs:", obs, "reward:", reward, "terminated:", terminated, "info:", info)

Initial observation: {'agent': array([0, 3]), 'target': array([3, 2])}
Initial info: {'distance': np.float64(4.0)}

After one step (action=0, 'right'):
obs: {'agent': array([0, 4]), 'target': array([3, 2])} reward: 0 terminated: False info: {'distance': np.float64(5.0)}


Registering the environment

In [15]:
# Register the environment so we can create it with gym.make()
gym.register(
    id="gymnasium_env/GridWorld-v0",
    entry_point=GridWorldEnv,
    max_episode_steps=300,  # Prevent infinite episodes
)
print("Registered!")
gym.pprint_registry()

Registered!
===== classic_control =====
Acrobot-v1             CartPole-v0            CartPole-v1
MountainCar-v0         MountainCarContinuous-v0 Pendulum-v1
===== phys2d =====
phys2d/CartPole-v0     phys2d/CartPole-v1     phys2d/Pendulum-v0
===== box2d =====
BipedalWalker-v3       BipedalWalkerHardcore-v3 CarRacing-v3
LunarLander-v3         LunarLanderContinuous-v3
===== toy_text =====
Blackjack-v1           CliffWalking-v1        CliffWalkingSlippery-v1
FrozenLake-v1          FrozenLake8x8-v1       Taxi-v4
===== tabular =====
tabular/Blackjack-v0   tabular/CliffWalking-v0
===== None =====
Ant-v2                 Ant-v3                 GymV21Environment-v0
GymV26Environment-v0   HalfCheetah-v2         HalfCheetah-v3
Hopper-v2              Hopper-v3              Humanoid-v2
Humanoid-v3            HumanoidStandup-v2     InvertedDoublePendulum-v2
InvertedPendulum-v2    Pusher-v2              Reacher-v2
Swimmer-v2             Swimmer-v3             Walker2d-v2
Walker2d-v3
===== mujoco ====

C:\Users\abudh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\gymnasium\envs\registration.py:637: UserWarning: WARN: Overriding environment gymnasium_env/GridWorld-v0 already in registry.
  logger.warn(f"Overriding environment {new_spec.id} already in registry.")


In [16]:
env = gym.make("gymnasium_env/GridWorld-v0", size=5)
print(env)

# Access the unwrapped instance to reach your custom attributes directly
print("Grid size:", env.unwrapped.size)

<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridWorldEnv<gymnasium_env/GridWorld-v0>>>>>
Grid size: 5


In [17]:
# Test specific action sequences to verify behavior
env = gym.make("gymnasium_env/GridWorld-v0")
obs, info = env.reset(seed=42)  # Use seed for reproducible testing

print(f"Starting position - Agent: {obs['agent']}, Target: {obs['target']}")

# Test each action type
actions = [0, 1, 2, 3]  # right, up, left, down
for action in actions:
    old_pos = obs['agent'].copy()
    obs, reward, terminated, truncated, info = env.step(action)
    new_pos = obs['agent']
    print(f"Action {action}: {old_pos} -> {new_pos}, reward={reward}")


Starting position - Agent: [0 3], Target: [3 2]
Action 0: [0 3] -> [0 4], reward=0
Action 1: [0 4] -> [0 4], reward=0
Action 2: [0 4] -> [0 3], reward=0
Action 3: [0 3] -> [1 3], reward=0


Check Environment Validity

In [18]:
from gymnasium.utils.env_checker import check_env

# This will catch many common issues
try:
    check_env(env)
    print("Environment passes all checks!")
except Exception as e:
    print(f"Environment has issues: {e}")

Environment passes all checks!


C:\Users\abudh\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\gymnasium\utils\env_checker.py:390: UserWarning: WARN: The environment (<TimeLimit<OrderEnforcing<PassiveEnvChecker<GridWorldEnv<gymnasium_env/GridWorld-v0>>>>>) is different from the unwrapped version (<GridWorldEnv<gymnasium_env/GridWorld-v0>>). This could effect the environment checker as the environment most likely has a wrapper applied to it. We recommend using the raw environment for `check_env` using `env.unwrapped`.
  logger.warn(


Using Wrappers

In [19]:
from gymnasium.wrappers import FlattenObservation

env = gym.make("gymnasium_env/GridWorld-v0")
print("Original observation_space:", env.observation_space)

wrapped_env = FlattenObservation(env)
print("Flattened observation_space:", wrapped_env.observation_space)

obs, info = wrapped_env.reset(seed=0)
print("Flattened obs (agent_x, agent_y, target_x, target_y):", obs)

Original observation_space: Dict('agent': Box(0, 4, (2,), int64), 'target': Box(0, 4, (2,), int64))
Flattened observation_space: Box(0, 4, (4,), int64)
Flattened obs (agent_x, agent_y, target_x, target_y): [4 3 2 1]


In [20]:
vec_env = gym.make_vec("gymnasium_env/GridWorld-v0", num_envs=3)
print(vec_env)

obs, info = vec_env.reset(seed=0)
print("Batched obs:", obs)

SyncVectorEnv(gymnasium_env/GridWorld-v0, num_envs=3)
Batched obs: {'agent': array([[4, 3],
       [2, 2],
       [4, 1]]), 'target': array([[2, 1],
       [3, 4],
       [0, 1]])}
